### Phase 2: Data Cleaning & Standardization

#### Environment Setup

In [0]:
df_sales = spark.read.table("bronze_sales")
df_products = spark.read.table("bronze_products")
df_stores = spark.read.table("bronze_stores")

print("Initialized: Bronze tables loaded successfully!")

#### Spot Missing Values

In [0]:
from pyspark.sql.functions import col, count, when

print("Missing Values in Products Table:")
display(
    df_products.select(
        [count(when(col(c).isNull(), c)).alias(c)
         for c in df_products.columns]
    )
)

In [0]:
# Check missing values for the Sales table
print("Missing Values in Sales Table:")
display(
    df_sales.select(
        [count(when(col(c).isNull(), c)).alias(c)
         for c in df_sales.columns]
    )
)

In [0]:
# Check missing values for the Store table
print("Missing Values in Store Table:")
display(
    df_stores.select(
        [count(when(col(c).isNull(), c)).alias(c)
         for c in df_stores.columns]
    )
)

Since there are no null values in df_stores, we are good to proceed.

In [0]:
print("Sales Summary Statistics:")
df_sales.describe (['sales', 'revenue', 'stock', 'price']).display()

In [0]:
print("Products Summary Statistics:")
df_products.describe(['product_length', 'product_depth', 'product_width']).display()

#### Handling Missing Variable

In [0]:
# Impute missing product dimensions with their respective averages and set missing 'cluster_id' to 'Unknown'

from pyspark.sql.functions import mean

avg_length = df_products.agg(mean("product_length")).collect()[0][0]
avg_depth = df_products.agg(mean("product_depth")).collect()[0][0]
avg_width = df_products.agg(mean("product_width")).collect()[0][0]

df_products_clean = df_products.na.fill({
    "product_length": avg_length,
    "product_depth": avg_depth,
    "product_width": avg_width,
    "cluster_id": "Unknown"
})

# Capture the total number of products processed
total_products = df_products_clean.count()

print(f" Imputation Metrics:")
print(f"   -> Imputed missing lengths with mean: {avg_length:.2f} cm/in")
print(f"   -> Imputed missing depths with mean: {avg_depth:.2f} cm/in")
print(f"   -> Imputed missing widths with mean: {avg_width:.2f} cm/in")
print(f"   -> Structured 'cluster_id' missing text with placeholder: 'Unknown'")
print(f"   -> Total product records cataloged: {total_products}")

# Focus only on the keys we altered to make sure they filled down correctly
df_products_clean.select("product_id", "product_length", "product_depth", "product_width", "cluster_id").limit(5).display()

In [0]:
from pyspark.sql.functions import col, count, when

# We use approxQuantile because it is highly efficient on large Spark datasets
sales_median = df_sales.approxQuantile("sales", [0.5], 0.01)[0]
revenue_median = df_sales.approxQuantile("revenue", [0.5], 0.01)[0]
price_median = df_sales.approxQuantile("price", [0.5], 0.01)[0]

total_rows = df_sales.count()

df_sales_clean = df_sales.na.fill({
    "sales": sales_median,
    "revenue": revenue_median,
    "price": price_median
})

final_rows = df_sales_clean.count()
dropped_rows = total_rows - final_rows

print("\nPreviewing Imputed & Filtered Target Columns:")
df_sales_clean.select("date", "product_id", "sales", "revenue","price").limit(5).display()

#### Standardize Formats

In [0]:
# Formating Date
from pyspark.sql.functions import col, when, trim, lower, to_date, regexp_replace
from pyspark.sql.types import DoubleType, IntegerType

# Standardize date formatting safely using yyyy-MM-dd
df_sales_dates = df_sales.withColumn("date_parsed", to_date(col("date"), "yyyy-MM-dd"))


In [0]:
df_sales_currency = df_sales_dates \
    .withColumn("price_cleaned", regexp_replace(col("price"), r"[\$,]", "")) \
    .withColumn("revenue_cleaned", regexp_replace(col("revenue"), r"[\$,]", ""))

# Type Casting: Replaced try_cast with standard .cast() linked to explicit Spark Types
df_sales_standardized = df_sales_currency \
    .withColumn("price_cleaned", col("price_cleaned").cast(DoubleType())) \
    .withColumn("revenue_cleaned", col("revenue_cleaned").cast(DoubleType())) \
    .withColumn("sales", col("sales").cast(IntegerType())) \
    .withColumn("date_parsed", col("date_parsed"))

print("Visual Verification of Standardized Schema & Capped Outliers:")
df_sales_standardized.select("date_parsed", "price_cleaned", "revenue_cleaned", "sales").limit(5).display()

In [0]:
print("Outlier Profiling and Capping via IQR...")
# Compute Interquartile Range (IQR) safely
quantiles = df_sales_standardized.approxQuantile("sales", [0.25, 0.75], 0.05)
q1, q3 = quantiles[0], quantiles[1]
iqr = q3 - q1

lower_bound = q1 - (3.0 * iqr)
upper_bound = q3 + (3.0 * iqr)

# Handle anomalies by capping them at the boundary
df_sales_final_clean = df_sales_standardized.withColumn(
    "sales_capped",
    when(col("sales") > upper_bound, upper_bound)
    .when(col("sales") < lower_bound, lower_bound)
    .otherwise(col("sales"))
)

print(f"Outlier capping engine complete.")
print(f"Statistical Boundaries Applied -> Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")


#### Negative Sales Value Check

In [0]:
# Check if there are any negative values hiding anywhere
print("\n Previewing actual negative sales records (if any exist):")
df_sales_clean.filter(col("sales") < 0).limit(5).display()

In [0]:
df_stores.write.format("delta").mode("overwrite").saveAsTable("silver_stores")
df_sales_clean.write.format("delta").mode("overwrite").saveAsTable("silver_sales")
df_products_clean.write.format("delta").mode("overwrite").saveAsTable("silver_products")